In [0]:
-- Creamos el esquema (base de datos) para tu proyecto
CREATE SCHEMA IF NOT EXISTS datos_retail;
---- Le decimos a Databricks que use este catálogo por defecto
USE CATALOG workspace;
-- Le decimos a Databricks que use este esquema por defecto
USE datos_retail;

In [0]:
-- Creamos un volumen para guardar los archivos que escupa el scraper en Python
CREATE VOLUME IF NOT EXISTS raw_scraping_data;

In [0]:
-- Dimensión de los productos (Actualizada con nombre_producto)
CREATE TABLE IF NOT EXISTS datos_retail.dim_producto (
    id_producto BIGINT GENERATED ALWAYS AS IDENTITY, -- Autoincremental nativo
    sku_original STRING,
    competidor STRING,
    marca STRING,
    categoria STRING,
    nombre_producto STRING, -- 🚀 NUEVA: Para tooltips y detalle en Power BI
    url_producto STRING
) USING DELTA;

-- Tabla de hechos (Actualizada con cuotas)
CREATE TABLE IF NOT EXISTS datos_retail.fact_precios_diarios (
    id_hecho BIGINT GENERATED ALWAYS AS IDENTITY,
    id_producto BIGINT,
    fecha_extraccion DATE,
    precio_regular DOUBLE,
    precio_oferta DOUBLE,
    cuotas INT,             -- 🚀 NUEVA: Para analizar financiación
    hay_stock BOOLEAN
) USING DELTA
PARTITIONED BY (fecha_extraccion); -- Particionar por fecha optimiza las consultas de series de tiempo

In [0]:
import os
import glob
from pyspark.sql.functions import col, lit

# 1. Definir la ruta apuntando al Volumen de Databricks creado previamente
carpeta_data = "/Volumes/workspace/datos_retail/raw_scraping_data/" 

# 2. Buscar todos los archivos CSV con el patrón correcto (ajustado a la imagen)
patron_busqueda = os.path.join(carpeta_data, "fravega_*.csv")
archivos_csv = glob.glob(patron_busqueda)

if not archivos_csv:
    raise FileNotFoundError(f"❌ No se encontraron archivos CSV en la ruta: {carpeta_data}")

# 3. Obtener el archivo más reciente
archivo_mas_reciente = sorted(archivos_csv)[-1]
print(f"⏳ Archivo más reciente detectado: {archivo_mas_reciente}")

# 4. Extraer la fecha del nombre del archivo 
# Ej: "fravega_2026-10-29.csv" -> "2026-10-29"
nombre_archivo = os.path.basename(archivo_mas_reciente)
fecha_archivo = nombre_archivo.replace("fravega_", "").replace(".csv", "")

print(f"📅 Fecha extraída para la Tabla de Hechos: {fecha_archivo}")

# 5. Lectura del dataframe usando la ruta dinámica
print("⏳ Leyendo archivo CSV...")
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .load(archivo_mas_reciente)

# Forzamos los tipos correctos
df_bronze = df_bronze.withColumn("precio_regular", col("precio_regular").cast("double")) \
                     .withColumn("precio_oferta", col("precio_oferta").cast("double")) \
                     .withColumn("hay_stock", col("hay_stock").cast("boolean")) \
                     .withColumn("cuotas", col("cuotas").cast("int"))

# OPTIMIZACIÓN 1: Guardar en memoria para no leer el CSV dos veces
df_bronze.cache()
df_bronze.createOrReplaceTempView("datos_nuevos_temp")
print("✅ Datos en memoria. Ejecutando MERGE en Dimensión Producto...")

# 6. UPSERT en Dimensión Producto
spark.sql("""
    MERGE INTO datos_retail.dim_producto AS destino
    USING datos_nuevos_temp AS origen
    ON destino.sku_original = origen.sku_original AND destino.competidor = origen.competidor
    WHEN MATCHED THEN
        UPDATE SET 
            destino.marca = origen.marca,
            destino.categoria = origen.categoria,
            destino.url_producto = origen.url_producto,
            destino.nombre_producto = origen.nombre_producto
    WHEN NOT MATCHED THEN
        INSERT (sku_original, competidor, marca, categoria, url_producto, nombre_producto)
        VALUES (origen.sku_original, origen.competidor, origen.marca, origen.categoria, origen.url_producto, origen.nombre_producto)
""")

print("✅ Dimensión actualizada. Insertando snapshot en Tabla de Hechos...")

# 7. INSERT en Tabla de Hechos (Usando la fecha del archivo en lugar de current_date)
df_dim = spark.table("datos_retail.dim_producto")

df_fact = df_bronze.join(
    df_dim,
    on=["sku_original", "competidor"],
    how="inner"
).select(
    col("id_producto"),
    lit(fecha_archivo).cast("date").alias("fecha_extraccion"),
    col("precio_regular"),
    col("precio_oferta"),
    col("hay_stock"),
    col("cuotas")
)

# OPTIMIZACIÓN 2: Reemplazo dinámico de partición para volver el script idempotente (no duplica datos si lo corres 2 veces)
df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"fecha_extraccion = '{fecha_archivo}'") \
    .saveAsTable("datos_retail.fact_precios_diarios")

print("🏆 ¡PIPELINE DINÁMICO COMPLETADO! Power BI ya puede consumir los datos actualizados.")

In [0]:
SELECT * FROM datos_retail.dim_producto;

In [0]:
SELECT * FROM datos_retail.fact_precios_diarios;

In [0]:
TRUNCATE TABLE datos_retail.fact_precios_diarios;
TRUNCATE TABLE datos_retail.dim_producto;